# Block-to-Point Kriging of Corrected Rover Neutron Counts

**Yuxi Zhang** · Oct 25, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebook performs Block-to-Point Kriging (B2PK/BTPK), also referred to as Area-to-Point Kriging (A2PK/ATPK), on corrected CRNS rover neutron counts to generate a point-support map of corrected neutron counts. It can also perform Point-to-Point Kriging (P2PK), equivalent to Ordinary Kriging, when each block is represented by a single discretization point located at the block centroid.

⚙️ **Main Processing Steps**  

1. **Load data**  

   a. Load the block-support data from the corrected rover file, `{farm_name}_{date}_counts_corr.csv`, and convert it into a GeoPandas GeoDataFrame named `block_support`.
   b. Load the target point raster named `grid_target`. This is the target locations for generating maps.
   c. Load the target soil sample points from the soil samples and extract SiteID and geometry as `points_target`. This is the target locations for pairing soil samples.

2. **Discretize blocks**  

   a. Define the weighting function (w(x)), which represents the contribution of each discretized point to the total block value.

   b. Discretize the blocks. In this case, all blocks have the same size and shape: a circular COSMOS sensor footprint with a radius of 240 m. Each block is discretized into grid points with a spacing of 50 m within the footprint. The weights are normalized so that they sum to 1.

   Note: The diescritising grid spacing, weighting functions can be customised in the code blocks named 'B2PK Settings'.


   $$
   Z(S) = \mathbf{w}^{T}Z(\mathbf{X})
   $$

   where $\mathbf{X} $ and $ \mathbf{w} $ are column vectors representing the discretized point locations and their contribution weights, respectively:

   $$
   \mathbf{X} = [x_1, \dots, x_k]^T,
   \qquad
   \mathbf{w} = [w_1, \dots, w_k]^T.
   $$


3. **Calculate the block-support experimental semivariogram**  

   Generate the block-support experimental semivariogram (`p2p_model_exp`) from the `block_support` data using the `pyinterpolate` package.


4. **Sparsify the block data**  

   Sparsify the observed blocks (rover measurements) to reduce the risk of a singular or ill-conditioned kriging matrix. 
   a. Process the rover measurements in chronological order, discarding each new measurement if it lies within 50 m of any previously retained measurement.

   b. If required, split the data into training and testing sets. In this workflow, the retained rover measurements are used for training, while the discarded measurements are assigned to the test set. This split is intended to maximize data utilization and should not be regarded as a formal model validation. For formal validation, the retained measurements should be independently partitioned into training and testing sets, using a random sampling strategy.


5. **Deconvolve the point-support semivariogram**  

   a. Specify an initial estimate of the parameters of the point-support theoretical semivariogram (`b2b_model`).

   b. Calculate block-to-block semivariances from the point-support theoretical semivariogram. Pair the block-to-block semivariances with the distances between the corresponding block centroids (`p2p_dis_all` and `p2p_sem_all`) and compare them with `p2p_model_exp`.

   The block-to-block semivariance and covariance matrices (`b2b_sem_mat` and `b2b_cov_mat`) are also obtained during this process.

   c. Adjust the point-support theoretical semivariogram parameters specified in Step 5a until the resulting block-support theoretical semivariogram provides an appropriate fit to the block-support experimental semivariogram.
   
   d. Stop here if the objective is only to fit the variogram model.


6. **Prediction**  

   a. For each target location, calculate the block-to-point semivariance or covariance.

   b. Construct the Lagrange system and solve it to obtain the kriging-weight column vector (\boldsymbol{\lambda}) and the Lagrange multiplier (\mu):

   $$
   \boldsymbol{\lambda} = [\lambda_1, \dots, \lambda_k]^T,
   \qquad
   \mu \in \mathbb{R}.
   $$

7. **Create the raster**  
   7.1 Quality control.  
   7.2 Convert the point-support predictions into a raster and save the resulting map.  
   

8. **Visualize the results**  
   Visualize the interpolated neutron-count map and associated model outputs.

💬 **Notes**  



📥 **Input**

* `./data_lv2/{farm_name}/data/`
  * `{farm_name}_{date}_counts_corr.csv`: Processed rover data containing corrected neutron counts.
  * `target_grid.csv`: Gridded target points within the farm boundary, used as the B2PK prediction target locations.

📤 **Output**  
* `./data_lv2/{farm_name}/kriging/`
  * `kriged_b2pk/`: B2PK-interpolated neutron-count maps at point support.
  * `kriged_p2pk/`: P2PK-interpolated neutron-count maps at block support.

**Parameters**  
* Dataset preparation
  * `min_distance`: The minimum distance allowed between retained rover measurements, preventing points from being too close to one another. Measurements within `min_distance` of a previously retained measurement are discarded and excluded from kriging.
  * `train_split`: Selects one measurement from every `train_split` consecutive measurements for the training set. This subsampling is used only for model fitting and testing to reduce computational time. In a formal run, all retained measurements are used for kriging.
* Discritisizing blocks
  * `radius=240`: Radius of the COSMOS sensor footprint.
  * `calc_wr()`: Defines the horizontal weighting function. By default, an exponentially decaying weighting function is used.
  * `discretize_spacing=50`: Spacing between discretization points within the COSMOS footprint. Decreasing the spacing substantially increases the computational cost.
  
* Kriging system-related
  * `obs_error=3`: Observational error applied to the measurements in the kriging system.
  * `on_cov=True`: Determines whether the covariance-based kriging system is used. By contrast, the kriging system is solved on semivariance.
  
* Neighbour search strategy for Kriging
  * `nsectors=4`: Number of search sectors.
  * `nmax_total=24`: Maximum total number of neighbouring points used for kriging. Set to a very large value to disable this limit.
  * `nmax_per_sector=8`: Maximum number of neighbouring points used from each sector. Set to a very large value to disable this limit.
  * `search_range=variogram_range*1.0`: Search radius around each target location.


### Settings

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
import re
import rioxarray as rxr
from pyinterpolate import  ExperimentalVariogram, TheoreticalVariogram, build_theoretical_variogram

import datetime as dt
from shapely.geometry import Point, MultiPoint

save_pdf = False
display = True


In [ ]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc
buffer_dis = BUFFER_DIS


#### Pipeline parameters

In [ ]:
# Parameters 1
try:
    mode
except NameError:
    mode = 'b2pk'

# mode: b2pk, p2pk, b2pk fit, p2pk fit, b2pk test, p2pk test
# Fit mode is for fitting the variogram model, and test mode is for debuging and testing the code.

suffix = '_' + mode.replace(' ', '_')
print(f'Running {suffix} kriging mode.')

In [ ]:
dir_ok = dirs.lv2_ok
dir_krig = dirs.lv2_krig / f'kriged_{mode}'
dir_krig_variogram =dir_krig / 'variogram'
dir_krig_f = dir_krig / 'figures'
dir_krig.mkdir(parents=True, exist_ok=True)
dir_krig_variogram.mkdir(parents=True, exist_ok=True)
dir_krig_f.mkdir(parents=True, exist_ok=True)

varname = 'CountsCorr'

#### B2PK Settings

In [ ]:
radius = 240 # Cosmos footprint
min_distance = 50 # To avoid points that are too close to each other, which can cause numerical instability in kriging. Set to 0 to disable this filter.

on_cov = True
on_sem = not on_cov

save_pdf = False

if 'fit' in mode.lower():
    train_split = 3
elif 'test' in mode.lower():
    train_split = 10
else:
    train_split = 1


# Kriging settings
if 'p2pk' in mode.lower():
    obs_error = 3   # Observational noise. Or 10-6 as a ridge term to improve numerical stability.
    discretize_spacing = radius * 2
    def calc_wr(r):
        return 1

else:
    obs_error = 3   # Observational noise.
    discretize_spacing = 50

    def calc_wr(r, r86=radius):
        #Kohli 2026 Review Eq. 1.
        return np.exp(-2*r/r86)

    # def calc_wr(r):
    #     #Schron 2015 Eq 5
    #     return np.exp(-r/127)

# radius: discretization radius for each block (assumed circular). 240m for Cosmos footprint
# train_split: 1 in every <train_split> values is used for training. train_split=1 means all points are used for training.
# calc_wr: function to calculate discretization weighting function
# discretize_spacing: spacing for discretization grid (m)
# obs_error: Observational noise. Or 10-6 as a ridge term to improve numerical stability.
# min_distance: remove points that are too close to each other (< min_distance)
# train_split: 1 in every <train_split> values is used for training. train_split=1 means all points are used for training.
# on_cov/on_sem: # Choose whether to calculate on covariance or semivariance matrices

## 1. Load data

### Find all rover data and select date to process

In [ ]:
# Find all corrected counts files and dates
regex = re.compile(rf'{farm_name}_(.*)_counts_corr.csv')
filenames_rover = [f for f in os.listdir(dirs.lv2_data) if regex.match(f)]
filenames_rover.sort()
dates_str = [regex.match(f).group(1) for f in filenames_rover]
filenames_rover

In [ ]:
# Parameters 2
try:
    date_str
except NameError:
    date_str = dates_str[0]  # Index of the rover file to process


In [ ]:
filename = [f for f in filenames_rover if date_str in f][0]
date = dt.datetime.strptime(date_str, '%Y-%m-%d')
farm_name, date_str, _, _ = re.split( '_', filename)
print(f'Processing file: {filename} for date: {date_str}')

### Rover data

In [ ]:
# Read rover data (corrected counts)
df_counts = pd.read_csv(os.path.join(dirs.lv2_data, filename))
df_counts['geometry'] = gpd.points_from_xy(df_counts['LongDec'], df_counts['LatDec'], crs= '4326').to_crs(proj_crs)
df_counts['Date Time(UTC)'] = pd.to_datetime(df_counts['Date Time(UTC)'], format='%Y-%m-%d %H:%M:%S')

date_start = df_counts['Date Time(UTC)'].min()
date_end = df_counts['Date Time(UTC)'].max()

block_support = gpd.GeoDataFrame(df_counts, geometry='geometry')


In [ ]:
# Create block geometry to krige
block_support = block_support[[varname, 'geometry']]

# Create buffer area
block_extend = block_support.copy()
block_extend.geometry = block_extend.geometry.buffer(radius)

ax= block_extend.plot(alpha=0.3, color='b')
block_support.plot(ax=ax, alpha=0.3, color='r')

In [ ]:
# Read farm boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))

In [ ]:
# Read grid points (raster) as target locations for kriging. Convert to GeoDataFrame.
if ('fit' in mode.lower()) or ('test' in mode.lower()):
    filename_grid = f'target_grid_test.tif'

else:
    filename_grid = f'target_grid.tif'

grid_target_da = rxr.open_rasterio(os.path.join(dirs.lv2_data, filename_grid), mask_and_scale=True).squeeze()
grid_target = grid_target_da.to_dataframe(name='area', 
                                              dim_order=['y', 'x']).sort_index(ascending=[False, True]).reset_index()
grid_target['geometry'] = gpd.points_from_xy(grid_target['x'], grid_target['y'], crs=proj_crs)
grid_target = gpd.GeoDataFrame(grid_target, geometry='geometry', crs=proj_crs)

In [ ]:
# Read the soil sample geometry as target locations for kriging
# pth_soil = os.path.join(pth_out1_soil, f'{farm_name}_soilsamples_fillna.csv')
pth_soil = os.path.join(dirs.lv2_data, f'{farm_name}_soilsamples_weighted.csv')

df_soil = pd.read_csv(pth_soil)

geometry = gpd.points_from_xy(df_soil['Longitude'], df_soil['Latitude'], crs='EPSG:4326').to_crs(proj_crs)
gdf_soil = gpd.GeoDataFrame(df_soil, geometry=geometry, crs=proj_crs)

points_soil = gdf_soil[gdf_soil['Date'] == date_str]

points_target = points_soil[['SiteID', 'geometry']]
points_target['x'] = points_target.geometry.x
points_target['y'] = points_target.geometry.y

## 2. Descritize blocks
Descritize each block into multiple points around centroid for kriging with weights.  

In [ ]:
# Descritize each block into multiple points around centroid for kriging with weights.
def discretize_block(centroid, radius=radius, spacing=discretize_spacing):
    coords_X = []
    dis_X = []
    weights_X = []
    coords_mat = []
    weights_mat = []
    # Make sure to include 0
    interval = np.arange(- np.floor(radius/spacing) * spacing, 
                         (np.floor(radius/spacing)) * spacing +1, spacing)
    for i in interval:
        for j in interval:  
            dis_ij = np.sqrt(i**2 + j**2)
            weight = calc_wr(dis_ij)
            if (0 <= dis_ij <= radius) and (weight > 0):
                coords_X.append(Point(centroid.x + i, centroid.y + j))
                dis_X.append(dis_ij)
                weights_X.append(weight)
                weights_mat.append(weight)
            else:
                weights_mat.append(0)

            coords_mat.append(Point(centroid.x + i, centroid.y + j))

    # Create MultiPoint and array of coordinates
    coords_X = MultiPoint(coords_X)
    dis_X = np.array(dis_X)
    weights_X = np.array(weights_X) / np.sum(weights_X)
    weights_mat = np.array(weights_mat) / np.sum(weights_mat)
    weights_mat = weights_mat.reshape(len(interval), len(interval))
    # coords_mat = coords_mat.reshape(len(interval), len(interval))
    return coords_X, dis_X, weights_X, weights_mat



Apply discretization to all blocks.

In [ ]:
# Apply discretization to all blocks
block_support[['coords', 'dist', 'weights'
                ]] = block_support.apply(lambda row: discretize_block(row.geometry.centroid,
                radius=radius, spacing=discretize_spacing), axis=1).apply(pd.Series).iloc[:, :3]
# Index points for 1-len(block_support)
block_support.index = range(1, len(block_support) + 1)
block_support.index.name = 'idx'

# Apply discretization to a test block to obtain weights
coords_test, _, weights, weights_mat = discretize_block(Point(0, 0))


Visualize weighting function.

In [ ]:
# Visualize weighting function
for geom, weight in zip(coords_test.geoms, weights):
    x = geom.x
    y = geom.y
    z = weight
# Plot 3D weights
fig = plt.figure(figsize=(12, 6))
ax1 = fig.add_subplot(121, projection='3d')
# Create a surface plot
X = np.array([geom.x for geom in coords_test.geoms])
Y = np.array([geom.y for geom in coords_test.geoms])
Z = np.array([weight for weight in weights])

# ax1.scatter(xs=X, ys=Y, zs=Z, c='magenta', marker='o')
if len(X) > 0:
    dx = dy = discretize_spacing
    dz = Z
    ax1.bar3d(X.flatten()-dx/2, Y.flatten()-dy/2, np.zeros_like(Z), dx, dy, dz, color='pink', alpha=0.8)
    xmin, xmax = ax1.get_xlim()
    ymin, ymax = ax1.get_ylim()
    # Project a shadow on the XZ plane
    ax1.bar3d(X.flatten(), np.full_like(Y.flatten(), ymax), np.zeros_like(Z), dx, 0.01, dz, color='lightgrey', alpha=1)

    # Project a shadow on the YZ plane
    ax1.bar3d(np.full_like(X.flatten(), xmin), Y.flatten(), np.zeros_like(Z), 0.01, dy, dz, color='lightgrey', alpha=1)

    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(ymin, ymax)
    # ax1.view_init(elev=30, azim=45)
    # ax1.plot_trisurf(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.8)
else:
    print("Discretization resulted in only one point; skipping weight visualization.")

ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Weight')
plt.tight_layout()
# ax2 = fig.add_subplot(122)
# sc = ax2.scatter(X, Z, c= 'deeppink', s=100)



## 3. Calculate block-support experimental semivariance/covariance

In [ ]:
if farm_name == 'Nowley':
    extent_ratio =  3 / 4
elif farm_name == 'Willora':
    extent_ratio = 3 / 5
else:
    extent_ratio = 1/2

print(f'Using a ratio of {extent_ratio} for farm {farm_name} to determine variogram extent.')

In [ ]:
# Create datset to fit variogram model
block_ds = pd.DataFrame({'x': block_support.geometry.x, 'y': block_support.geometry.y, 'z': block_support[varname]}).to_numpy()
# ds
print((block_support.geometry.x.max() - block_support.geometry.x.min()), 
      (block_support.geometry.y.max() - block_support.geometry.y.min()))
extent = np.sqrt((block_support.geometry.x.max() - block_support.geometry.x.min())**2 + (block_support.geometry.y.max() - block_support.geometry.y.min())**2) * extent_ratio

In [ ]:
step_size = 50  # meters - remember, that this parameter is related to projection!
b2b_variogram_exp = ExperimentalVariogram(ds=block_ds, step_size=step_size, max_range=extent, 
                                               is_semivariance=True, is_covariance=True)
b2b_variogram_exp.plot(semivariance=True, covariance=True, variance=True)

## 4. Sparse blocks to avoid large area of overlap. 
Remove points that are too close to each other to avoid singular or ill-condition matrix when inversing the semivariance/covariance matrix.  
Train-test split is optional. If using all sparsed data for training, set `train_split` to 1.

In [ ]:
# Take points to a new collection one by one.
# If a new point is too close to all existing points, discard it. Else add it to the collection.
block_support_select = block_support.iloc[0:1, :].copy()

idx_select = []
for idx, row in block_support.iloc[1:, :].iterrows():
    dists = block_support_select.geometry.distance(row.geometry)
    if (dists < min_distance).any():
        continue
    else:
        block_support_select = pd.concat([block_support_select, row.to_frame().T], ignore_index=False)
        idx_select.append(idx)

block_support_select = block_support.loc[idx_select].copy()
print(f'Removed {len(block_support) - len(block_support_select)}/{len(block_support)} points that are closer than {min_distance} m to other points.')


# Train-test split in selected points
block_support_train = block_support_select.iloc[::train_split, :].copy()   # Take every second point for training
block_support_test = block_support[~block_support.index.isin(block_support_train.index)]
print(f'Training points: {len(block_support_train)}, Test points: {len(block_support_test)}')


# Check the selected and removed points
plt.figure()
ax = bound.plot(edgecolor='black', facecolor='none')
# block_support.plot(ax=ax, color='grey', alpha=0.5, label='All points')
block_support_train.plot(ax=ax, color='green', marker='s', alpha=0.5, label='Prediction set')
block_support_test.plot(ax=ax, color='deeppink', marker='o', alpha=0.5, label='Evaluation set')
plt.legend()

## 5. Deconvolute P2P theoretical model

### 5a. Build P2P theoretical model
* Initial guess and adjustment.

In [ ]:
# Initial automatic variogram fitting with all data pairs within the extent
models = ['linear'] # 'linear', 'power', 'gaussian', 'spherical', 'exponential'
p2p_model = TheoreticalVariogram()
p2p_model.autofit(
    experimental_variogram=b2b_variogram_exp,
    models_group=models
)

variogram_range = p2p_model.rang
nugget = p2p_model.nugget
partial_sill = p2p_model.sill
sill = p2p_model.sill + nugget
model = p2p_model.model_type

print(f'Nugget={nugget:.2f}, Range={variogram_range}, Sill={sill:.2f}, Partial Sill={partial_sill:.2f}, Spatial Dependency Index (Nugget/Sill*100)={nugget/sill*100:.2f}')

p2p_model.plot()


### 5c. Maually adjust the P2P theoretical model below

In [ ]:
# Adjust nugguet to fit P2P variogram so that B2B variogram fits real data.
# variogram_range = 1500
nugget = np.quantile(b2b_variogram_exp.semivariances[b2b_variogram_exp.lags <= variogram_range], 0.01)
sill = b2b_variogram_exp.semivariances[b2b_variogram_exp.lags > variogram_range].mean()
# nugget = 460

print(f'Nugget={nugget:.2f}, Range={variogram_range}, Sill={sill:.2f}, Partial Sill={sill-nugget:.2f}, Spatial Dependency Index (Nugget/Sill*100)={nugget/sill*100:.2f}')
# print(b2b_variogram_exp)

p2p_model = build_theoretical_variogram(
    experimental_variogram=b2b_variogram_exp,  # This parameter is actually partial sill, not total sill. Set to 0 to force fitting through the origin.
    models_group=model, # 'circular', 'cubic', 'exponential', 'gaussian', 'linear', 'power', 'spherical', 'all'
    nugget=nugget,
    rang=variogram_range,
    sill=max(0.00001, sill-nugget),
)
p2p_model.plot()
p2p_model

# if model=='gaussian':
#     variogram_range = variogram_range * 2

#### Save or load the fitted variogram model

This cell saves the fitted variogram model or loads a previously fitted model.

* If `mode` is set to `'fit'`, the currently fitted variogram model is saved to file.
* For all other modes, a previously saved variogram model is loaded from file. It prioritizes loading a variogram model that has been manually saved with the `'_saved'` suffix.

In [ ]:
# If the mode is 'fit', save the variogram model. Else load from file.
file_variogram =dir_krig_variogram / f'{farm_name}_{date_str}_{varname}_p2p_variogram.json'
file_variogram_saved = file_variogram.with_name(file_variogram.name.replace('.json', '_saved.json'))


if 'fit' in mode.lower():
    # # Export the theoretical model
    p2p_model.to_json(file_variogram)
    print(f'Saved variogram model to file: {file_variogram_saved}')
else:
    p2p_model = TheoreticalVariogram()
    if os.path.exists(file_variogram_saved):
        p2p_model.from_json(file_variogram_saved)
        nugget = p2p_model.nugget
        sill = p2p_model.sill + p2p_model.nugget
        variogram_range = p2p_model.rang
        model = p2p_model.model_type
        print(f'Loaded variogram model from saved file: Nugget={nugget:.2f}, Range={variogram_range}, Sill={sill:.2f}, Partial Sill={sill-nugget:.2f}, Nugget/Sill={nugget/sill:.2f}')

    elif os.path.exists(file_variogram):
        print('Loading variogram model from last run.')
        p2p_model.from_json(file_variogram)
        nugget = p2p_model.nugget
        sill = p2p_model.sill + p2p_model.nugget
        variogram_range = p2p_model.rang
        model = p2p_model.model_type
        print(f'Loaded variogram model from last run: Nugget={nugget:.2f}, Range={variogram_range}, Sill={sill:.2f}, Partial Sill={sill-nugget:.2f}, Nugget/Sill={nugget/sill:.2f}')

    else:
        print('Previous variogram model file not found. Using newly fitted model.')
        pass


In [ ]:
x0 = b2b_variogram_exp.lags
y0_sem = b2b_variogram_exp.semivariances

plt.figure(figsize=(4, 3))
ax = plt.gca()
ax.plot(x0, y0_sem, ls='', marker='o', ms=8, c='green', alpha=0.6, label ='Empirical Semivariance')

# ax.plot(x0, y0_cov, ls='', marker='+', ms=5, c='red', label ='Empirical Covariance')
# ax.plot(x0, sill-y0_sem, ls='', marker='+', ms=5, c='blue', label ='Empirical Covariance')
ax.set_xlabel('Distance (m)')
ax.set_ylabel('Theoretical Covariance (B2B)')
plt.legend(loc='lower right')

There are four levels of spatial dependency.

| Lower Limit (included)       | Upper Limit (excluded) | Strength              |
|------------------------------|------------------------|-----------------------|
| 0                            | 25                     | strong                |
| 25                           | 75                     | moderate              |
| 75                           | 95                     | weak                  |
| 95                           | inf                    | no spatial dependence |


In [ ]:
p2p_model

### 5b. Calculate block to block semivariance
As the blocks are of the same shape, size and discritization, the discritizing weights $\mathbf{w}$ only need to be calculated once.

In [ ]:
counts = len(weights)
arr_wts_ij = []
for i in range(counts): # Make sure the loop is exactly the  same as in B2B semivariance calculation
    for j in range(counts):
        arr_wts_ij.append(weights[i] * weights[j])
arr_wts_ij_norm = arr_wts_ij/np.sum(arr_wts_ij)

In [ ]:
# Within-block semivariance (b2b_sem_0)
block_i = block_support_train.iloc[0]
arr_dis = []
for xi in block_i['coords'].geoms:
    for xj in block_i['coords'].geoms:
        arr_dis.append(xi.distance(xj))

arr_dis = np.array(arr_dis)
b2b_dis_0 = np.dot(arr_wts_ij_norm, arr_dis)
b2b_sem_0 = np.dot(arr_wts_ij_norm, p2p_model.predict(arr_dis))
print(b2b_sem_0)

In [ ]:
start_time = dt.datetime.now()

p2p_dis_all = []
b2b_dis_all = []
p2p_sem_all = []
b2b_sem_all = []
b2b_cov_all = []
p2p_cov_all = []
b2b_sem_mat = np.zeros((len(block_support_train) + 1, len(block_support_train) + 1))
b2b_cov_mat = np.zeros((len(block_support_train) + 1, len(block_support_train) + 1))

# Block to block semivariance
for i in range(len(block_support_train)):
    for j in range(i, len(block_support_train)):
        block_i = block_support_train.iloc[i]
        block_j = block_support_train.iloc[j]
        # counts_i = len(block_i['coords'])
        # counts_j = len(block_j['coords'])
        p2p_dis = block_i.geometry.distance(block_j.geometry)
        p2p_sem = p2p_model.predict(np.array([p2p_dis]))[0]
        p2p_cov = sill - p2p_sem
        
        # Block to block distance and semivariance
        arr_dis = []
        for xi in block_i['coords'].geoms:
            for xj in block_j['coords'].geoms:
                arr_dis.append(xi.distance(xj))

        arr_dis = np.array(arr_dis)
        b2b_dis = np.dot(arr_wts_ij_norm, arr_dis)

        if on_sem:
            b2b_sem = np.dot(arr_wts_ij_norm, p2p_model.predict(arr_dis))
            b2b_sem_mat[i, j] = b2b_sem
            b2b_sem_mat[j, i] = b2b_sem
            p2p_sem_all.append(p2p_sem)
            b2b_sem_all.append(b2b_sem)
        if on_cov:
            b2b_cov = np.dot(arr_wts_ij_norm, sill - p2p_model.predict(arr_dis))
            b2b_cov_mat[i, j] = b2b_cov
            b2b_cov_mat[j, i] = b2b_cov
            p2p_cov_all.append(p2p_cov)
            b2b_cov_all.append(b2b_cov)

        p2p_dis_all.append(p2p_dis)
        b2b_dis_all.append(b2b_dis)

end_time = dt.datetime.now()

time_delta = end_time - start_time
print("Process time:", time_delta.seconds/60, "minutes")


In [ ]:
if on_cov:
    try:
        np.linalg.cholesky(b2b_cov_mat + obs_error * np.eye(b2b_cov_mat.shape[0]))
        print("Cholesky decomposition successful.")
    except np.linalg.LinAlgError:
        print("Cholesky decomposition failed.")

In [ ]:
if on_sem:
    b2b_sem_0 = b2b_sem_mat[0][0]
    b2b_sem_mat = b2b_sem_mat - b2b_sem_0
    b2b_sem_all = np.array(b2b_sem_all) - b2b_sem_0

    # Set last row and column for Lagrange multiplier
    b2b_sem_mat[:, -1] = 1
    b2b_sem_mat[-1, :] = 1
    b2b_sem_mat[-1, -1] = 0

#### Plot B2B and P2P semivariace.

In [ ]:
x0 = b2b_variogram_exp.lags - step_size/2
y0_sem = b2b_variogram_exp.semivariances
df_variogram_empirical = pd.DataFrame({'distance': x0, 'p2p_semivariance': y0_sem})
df_variogram_empirical.to_csv(os.path.join(dir_krig_variogram, f'{farm_name}_{date_str}_{varname}_variogram_empirical.csv'), index=False)

p2p_dis_all = np.array(p2p_dis_all)
b2b_dis_all = np.array(b2b_dis_all)

if on_cov:
    p2p_cov_all = np.array(p2p_cov_all)
    b2b_cov_all = np.array(b2b_cov_all)
    p2p_sem_all = sill - p2p_cov_all
    b2b_sem_all = sill- b2b_cov_all

elif on_sem:
    p2p_sem_all = np.array(p2p_sem_all)
    b2b_sem_all = np.array(b2b_sem_all)
    p2p_cov_all = sill - p2p_sem_all
    b2b_cov_all = sill - b2b_sem_all

df_variogram = pd.DataFrame({'distance': p2p_dis_all, 'b2b_distance': b2b_dis_all,
                            'p2p_semivariance': p2p_sem_all, 'b2b_semivariance': b2b_sem_all,
                            'p2p_covariance': p2p_cov_all, 'b2b_covariance': b2b_cov_all})

df_variogram = df_variogram.dropna()
df_variogram = df_variogram.sort_values('distance')
# Exclude pairs that are farther than the variogram extent to focus on the range where the model is fitted.
df_variogram = df_variogram[df_variogram['distance'] <= extent]
df_variogram.to_csv(os.path.join(dir_krig_variogram, f'{farm_name}_{date_str}_{varname}_variogram_theoretical.csv'), index=False)

# Nugget offset: the difference between b2b and p2p semivariance at near-zero distance.
nugget_offset = df_variogram[df_variogram['distance']  < 1e-6].mean(numeric_only=True)
nugget_offset =nugget_offset['b2b_semivariance'] -  nugget_offset['p2p_semivariance']
print('Difference between b2b and p2p semivariance at near-zero distance (nugget offset):', nugget_offset)

In [ ]:
plt.figure(figsize=(4, 3))
ax = plt.gca()
ax.plot(df_variogram_empirical['distance'], df_variogram_empirical['p2p_semivariance'], ls='', marker='o', ms=8, c='green', alpha=0.6, label ='Empirical Semivariance')
ax.plot(df_variogram['distance'], df_variogram['p2p_semivariance'], ls='-', lw=2, marker='', ms=1, c='black', label ='Theoretical P2P Semivariance')
ax.plot(df_variogram['distance'], df_variogram['b2b_semivariance'], ls='', marker='o', ms=2, c='deeppink', label ='Theoretical B2B Semivariance')

ax.set_xlabel('Lag distance (m)')
ax.set_ylabel('Semivariance')
ax.set_title(f'{farm_name} {date_str} {varname}')
plt.legend(loc='lower right')

# Save figure
plt.savefig(dir_krig_variogram / f'{farm_name}_{date_str}_{varname}_variogram_comparison.png', bbox_inches='tight')
if save_pdf:
    plt.savefig(dir_krig_f / f'{farm_name}_{date_str}_{varname}_variogram_comparison.pdf', bbox_inches='tight')
plt.show()

In [ ]:
df_variogram['bin'] = pd.cut(df_variogram['distance'], bins=np.arange(0, extent + step_size, step_size))
df_variogram_bin = df_variogram.groupby('bin').agg({'distance': 'mean',
                                                    'p2p_semivariance': 'mean',
                                                    'b2b_semivariance': 'mean',
                                                    'p2p_covariance': 'mean',
                                                    'b2b_covariance': 'mean'}).reset_index()

df_variogram_bin['lag'] = df_variogram_bin['bin'].apply(lambda x: x.mid)
df_empirical = pd.DataFrame({'lag': b2b_variogram_exp.lags, 'b2b_semivariance': b2b_variogram_exp.semivariances})
df_variogram_bin = df_variogram_bin.merge(df_empirical, on='lag', how='left', suffixes=('_theoretical', '_empirical'))
df_variogram_bin = df_variogram_bin.dropna(how='any')
df_variogram_bin.to_csv(dir_krig_variogram / f'{farm_name}_{date_str}_{varname}_variogram_theoretical_binned.csv', index=False)
# df_variogram_bin

In [ ]:
plt.figure(figsize=(4, 3))
ax = plt.gca()
ax.plot(df_variogram_bin['lag'], df_variogram_bin['b2b_semivariance_empirical'], ls='', marker='o', ms=8, c='green', alpha=0.6, label ='Empirical Semivariance')
ax.plot(df_variogram_bin['lag'], df_variogram_bin['p2p_semivariance'], ls='-', lw=2, marker='', ms=1, c='black', label ='Theoretical P2P Semivariance')
ax.plot(df_variogram_bin['lag'], df_variogram_bin['b2b_semivariance_theoretical'], dashes=(1, 0.3), lw=5, marker='', ms=2, c='deeppink', label ='Theoretical B2B Semivariance')

ax.set_xlabel('Lag distance (m)')
ax.set_ylabel('Semivariance')
ax.set_title(f'{farm_name} {date_str} {varname} Variogram')
plt.legend(loc='lower right')

rmse = np.sqrt(np.mean((df_variogram_bin['b2b_semivariance_theoretical'] - df_variogram_bin['b2b_semivariance_empirical'])**2))
print(f'RMSE between B2B empirical and theoretical semivariance\nRMSE={rmse:.2f}, Nugguet={nugget:.2f}, Range={variogram_range}, Sill={sill:.2f}, Partial Sill={sill-nugget:.2f}, Nugget/Sill={nugget/sill:.2f}, extent={extent:.2f}, step_size={step_size}')


### 5d. Stop here if the objective is only to fit the variogram model.

* The notebook automatically stops at this point during a variogram-fitting run, where `mode = 'b2pk fit'` or `'p2pk fit'`.
* During a formal run, where `mode = 'b2pk'` or `'p2pk'`, predictions are generated in the subsequent sections.


In [ ]:
if 'fit' in mode.lower():
    import sys
    sys.exit("Stopping execution as mode='fit'")

## 6. Prediction

In [ ]:
arr_wts_j = weights.copy() # Weights are already normalized
counts_j = len(weights)

In [ ]:
# This defines the neighbour search strategy for kriging. 
# The search area is divided into sectors, and a maximum number of points is selected from each sector.
# This helps to ensure the stability of the kriging system 
# and that the kriging prediction is based on a representative sample of points from different directions around the target location.

nsectors = 4 # Number of search sectors
nmax_total = 24 # Maximum total number of neighbouring points used for kriging. Set to a very large value to disable this limit.
nmax_per_sector = 8 # Maximum number of points to consider in each sector. Set to a very large value to disable this filter.
search_range = variogram_range * 1.0 # Search radius around each target location.

### 6.1 Predict across the entire farm at the target grid locations

In [ ]:
# For mapping
# New code, assuming all blocks have the same shape and size
b2p_dis_all = []
b2p_sem_all = []
b2p_cov_all = []

b2p_dis_centroid_all = []
preds = []
preds_var = []
lambda_vals = []
mu_vals = []
# Unknown points to predict
points_failed = []
for m in range(len(grid_target)):
    point = grid_target.iloc[m]
    b2p_sem_mat = []
    b2p_cov_mat = []
    values = [] 
    row_idx = []    

    ############################ Selecting blocks based on distance and sector ############################
    # Divide the geographic azimuth into `nsectors` sectors. Moving clockwise, select the nearest zero point from each sector in turn, until either a total of `nmax_total` points has been selected, or each sector has reached its limit of `nmax_per_sector` points.

    # Calculate distance from point to block centroids and filter blocks within search range
    distance = block_support_train.geometry.distance(point.geometry)
    # Filter blocks within search range
    distance = distance[distance <= search_range] if search_range is not None else distance
    sector_rad = np.arctan2(block_support_train.geometry.x - point.geometry.x, block_support_train.geometry.y - point.geometry.y) # GPS angle in radians. 
    sector_angle =(sector_rad * 180 / np.pi ) % 360 # GPS angle in degrees. Clockwise, with 0 at north, 90 at east, -90 at west, and 180/-180 at south.
    sector = (sector_angle / 360 * nsectors).astype(int) # 0, 1, 2, 3, ..., nsectors-1

    # Take the closest max_neighbors blocks within search range, with a maximum of nmax_per_sector blocks per sector
    selected_blocks_idx = []
    for sec in sector.unique():
        # Filter distance and corresponding blocks in the same sector
        distance_sub = distance[sector == sec]
        # Return the indices of the ranked distance. Note that this is the iloc index, not the loc index.
        order_sub = np.argsort(distance_sub.values)
        # Find the loc indices of the first n values in the sorked distance_sub array
        idx = distance_sub.iloc[order_sub][:nmax_per_sector].index
        # if len(idx)<nmax_per_sector, fill the rest with -99
        if len(idx) < nmax_per_sector:
            idx = idx.tolist() + [-99] * (nmax_per_sector - len(idx))
        selected_blocks_idx.append(idx)

    # Create 2D array from 2D list of selected block indices. 
    selected_blocks_idx = np.array(selected_blocks_idx)
    # Transpose and flatten to make it a column vector. Remove -99 values which indicate no block selected in that sector.
    selected_blocks_idx = selected_blocks_idx.T
    selected_blocks_idx = selected_blocks_idx.flatten()
    selected_blocks_idx = selected_blocks_idx[selected_blocks_idx != -99]
    # Take the first nmax_total numbers. This is the final list of selected block indices for the current point.
    selected_blocks_idx = selected_blocks_idx[:nmax_total]
    ############################ Selecting blocks based on distance and sector ############################

    # Known blocks
    for j in range(len(block_support_train)):
        block_j = block_support_train.iloc[j]
        # If index not in selected_blocks_idx, skip
        if block_j.name not in selected_blocks_idx:
            continue

        # Calculate centroid distance between block and point
        b2p_dis_centroid = point.geometry.distance(block_j.geometry)

        # Block to point distance and semivariance
        arr_dis = []
        for xj in block_j['coords'].geoms:
            dis_j = point.geometry.distance(xj)
            arr_dis.append(dis_j)

        arr_dis = np.array(arr_dis)
        b2p_dis = np.dot(arr_wts_j, arr_dis)

        b2p_sem = np.dot(arr_wts_j, p2p_model.predict(arr_dis))
        b2p_cov = np.dot(arr_wts_j, sill - p2p_model.predict(arr_dis))

        # b2p_dis_all.append(b2p_dis)
        # b2p_sem_all.append(b2p_sem)
        # b2p_cov_all.append(b2p_cov)
        b2p_dis_centroid_all.append(b2p_dis_centroid)
        b2p_sem_mat.append(b2p_sem)
        b2p_cov_mat.append(b2p_cov)
        row_idx.append(j)
        values.append(block_j[varname])

    values = np.array(values)
    if len(row_idx) == 0:
        preds.append(np.nan)
        preds_var.append(np.nan)
        lambda_vals.append(np.nan)
        mu_vals.append(np.nan)
        continue

    row_idx.append(len(block_support))
    row_idx = np.array(row_idx)
    if on_sem: # Using semivariance matrix
        # Add Lagrange multiplier row
        b2p_sem_mat.append(1)
        b2p_sem_mat = np.matrix(b2p_sem_mat).T
        b2b_sem_mat_sub = b2b_sem_mat[row_idx][:, row_idx] 
        
        # Calculate kriging weights
        w = np.linalg.inv(b2b_sem_mat_sub) * b2p_sem_mat
        pred = np.dot(values, w[:-1])

    if on_cov: # Using covariance matrix
        row_idx = np.array(row_idx)[:-1]
        K = b2b_cov_mat[row_idx][:, row_idx]
        k = np.matrix(b2p_cov_mat).T

        # K = b2b_cov_mat_sub
        # k = b2p_cov_mat
        # Cholesky decomposition
        try:
            L = np.linalg.cholesky(K + obs_error * np.eye(K.shape[0]))
        except np.linalg.LinAlgError:
            # print(f"Cholesky decomposition failed for point {m}. Skipping this point.")
            points_failed.append(m)
            lambda_vals.append(np.nan)
            mu_vals.append(np.nan)
            preds.append(np.nan)
            preds_var.append(np.nan)
            continue

        def Kinv_b(b):
            y = np.linalg.solve(L, b)
            return np.linalg.solve(L.T, y)

        ones = np.matrix(np.ones(K.shape[0])).T
        Kinv1 = Kinv_b(ones)
        Kinvk = Kinv_b(k)
        v_mu = (ones.T @ Kinvk - 1)/(ones.T @ Kinv1)
        v_lambda = Kinvk - Kinv1 * v_mu
        pred = values @ v_lambda
        pred_var = v_lambda.T @ ((K + obs_error * np.eye(K.shape[0])) @ v_lambda - 2 * k) + (sill-nugget)

        lambda_vals.append(v_lambda)
        mu_vals.append(v_mu[0, 0])
        preds.append(pred[0, 0])
        preds_var.append(pred_var[0, 0])

    # break


### Check

Use this figure to identify any problematic locations where predictions could not be obtained. If there are many problematic locations, adjust the settings, such as `obs_error` and the neighbour-search strategy.


In [ ]:
# Check the selected and removed points
plt.figure()
ax = bound.plot(edgecolor='black', facecolor='none')
# block_support.plot(ax=ax, color='grey', alpha=0.5, label='All points')
block_support_train.buffer(search_range).plot(ax=ax, color='green', marker='s', alpha=0.1, label='Prediction set')
grid_target.iloc[points_failed].plot(ax=ax, color='blue', marker='v', alpha=0.2, label='Failed target point')
plt.legend()

### 6.2 Predict at soil samples location for calibration purpose

In [ ]:
### For soil sampling locations
# New code, assuming all blocks have the same shape and size
b2p_dis_all = []
b2p_sem_all = []
b2p_cov_all = []

b2p_dis_centroid_all = []
preds2 = []
preds_var2 = []
lambda_vals2 = []
mu_vals2 = []
points_failed = []

for m in range(len(points_target)):
    point = points_target.iloc[m]
    b2p_sem_mat = []
    b2p_cov_mat = []
    values = [] 
    row_idx = []


    ############################ Selecting blocks based on distance and sector ############################
    # Divide the geographic azimuth into `nsectors` sectors. Moving clockwise, select the nearest zero point from each sector in turn, until either a total of `nmax_total` points has been selected, or each sector has reached its limit of `nmax_per_sector` points.
    
    # Calculate distance from point to block centroids and filter blocks within search range
    distance = block_support_train.geometry.distance(point.geometry)
    # Filter blocks within search range
    distance = distance[distance <= search_range] if search_range is not None else distance
    sector_rad = np.arctan2(block_support_train.geometry.x - point.geometry.x, block_support_train.geometry.y - point.geometry.y) # GPS angle in radians. 
    sector_angle =(sector_rad * 180 / np.pi ) % 360 # GPS angle in degrees. Clockwise, with 0 at north, 90 at east, -90 at west, and 180/-180 at south.
    sector = (sector_angle / 360 * nsectors).astype(int) # 0, 1, 2, 3, ..., nsectors-1

    # Take the closest max_neighbors blocks within search range, with a maximum of nmax_per_sector blocks per sector
    selected_blocks_idx = []
    for sec in sector.unique():
        # Filter distance and corresponding blocks in the same sector
        distance_sub = distance[sector == sec]
        # Return the indices of the ranked distance. Note that this is the iloc index, not the loc index.
        order_sub = np.argsort(distance_sub.values)
        # Find the loc indices of the first n values in the sorked distance_sub array
        idx = distance_sub.iloc[order_sub][:nmax_per_sector].index
        # if len(idx)<nmax_per_sector, fill the rest with -99
        if len(idx) < nmax_per_sector:
            idx = idx.tolist() + [-99] * (nmax_per_sector - len(idx))
        selected_blocks_idx.append(idx)
    
    # Create 2D array from 2D list of selected block indices. 
    selected_blocks_idx = np.array(selected_blocks_idx)
    # Transpose and flatten to make it a column vector. Remove -99 values which indicate no block selected in that sector.
    selected_blocks_idx = selected_blocks_idx.T
    selected_blocks_idx = selected_blocks_idx.flatten()
    selected_blocks_idx = selected_blocks_idx[selected_blocks_idx != -99]
    # Take the first nmax_total numbers. This is the final list of selected block indices for the current point.
    selected_blocks_idx = selected_blocks_idx[:nmax_total]
    ############################ Selecting blocks based on distance and sector ############################

    # Known blocks
    for j in range(len(block_support_train)):
        block_j = block_support_train.iloc[j]
        # If index not in selected_blocks_idx, skip
        if block_j.name not in selected_blocks_idx:
            continue

        # Calculate centroid distance between block and point
        b2p_dis_centroid = point.geometry.distance(block_j.geometry)

        # Block to point distance and semivariance
        arr_dis = []
        for xj in block_j['coords'].geoms:
            dis_j = point.geometry.distance(xj)
            arr_dis.append(dis_j)

        arr_dis = np.array(arr_dis)
        b2p_dis = np.dot(arr_wts_j, arr_dis)

        b2p_sem = np.dot(arr_wts_j, p2p_model.predict(arr_dis))
        b2p_cov = np.dot(arr_wts_j, sill - p2p_model.predict(arr_dis))

        # b2p_dis_all.append(b2p_dis)
        # b2p_sem_all.append(b2p_sem)
        # b2p_cov_all.append(b2p_cov)
        b2p_dis_centroid_all.append(b2p_dis_centroid)
        b2p_sem_mat.append(b2p_sem)
        b2p_cov_mat.append(b2p_cov)
        row_idx.append(j)
        values.append(block_j[varname])

    values = np.array(values)
    if len(row_idx) == 0:
        preds2.append(np.nan)
        preds_var2.append(np.nan)
        lambda_vals2.append(np.nan)
        mu_vals2.append(np.nan)
        continue

    row_idx.append(len(block_support))
    row_idx = np.array(row_idx)
    if on_sem: # Using semivariance matrix
        # Add Lagrange multiplier row
        b2p_sem_mat.append(1)
        b2p_sem_mat = np.matrix(b2p_sem_mat).T
        b2b_sem_mat_sub = b2b_sem_mat[row_idx][:, row_idx] 
        
        # Calculate kriging weights
        w = np.linalg.inv(b2b_sem_mat_sub) * b2p_sem_mat
        pred = np.dot(values, w[:-1])
        preds2.append(pred[0, 0])

    if on_cov: # Using covariance matrix
        row_idx = np.array(row_idx)[:-1]
        K = b2b_cov_mat[row_idx][:, row_idx]
        k = np.matrix(b2p_cov_mat).T

        # K = b2b_cov_mat_sub
        # k = b2p_cov_mat
        # Cholesky decomposition
        try:
            L = np.linalg.cholesky(K + obs_error * np.eye(K.shape[0]))
        except np.linalg.LinAlgError:
            # print(f"Cholesky decomposition failed for point {m}. Skipping this point.")
            points_failed.append(m)
            lambda_vals2.append(np.nan)
            mu_vals2.append(np.nan)
            preds2.append(np.nan)
            preds_var2.append(np.nan)
            continue

        def Kinv_b(b):
            y = np.linalg.solve(L, b)
            return np.linalg.solve(L.T, y)

        ones = np.matrix(np.ones(K.shape[0])).T
        Kinv1 = Kinv_b(ones)
        Kinvk = Kinv_b(k)
        v_mu = (ones.T @ Kinvk - 1)/(ones.T @ Kinv1)
        v_lambda = Kinvk - Kinv1 * v_mu
        pred = values @ v_lambda
        pred_var = v_lambda.T @ ((K + obs_error * np.eye(K.shape[0])) @ v_lambda - 2 * k) + (sill-nugget)

        preds2.append(pred[0, 0])
        preds_var2.append(pred_var[0, 0])
        lambda_vals2.append(v_lambda)
        mu_vals2.append(v_mu[0, 0])
    # break

In [ ]:
if display:
    if on_cov:
        fig, axes = plt.subplots(1,2, figsize=(12,6))
        vmax = max(K.max(), k.max())
        vmin = min(K.min(), k.min())
        axes[0].imshow(K + obs_error * np.eye(K.shape[0]), cmap='RdYlGn', vmin=vmin, vmax=vmax)
        axes[1].imshow(k, cmap='RdYlGn', vmin=vmin, vmax=vmax)
        # Show colorbar
        # fig.colorbar(axes[0].images[0], ax=axes[0])
        fig.colorbar(axes[1].images[0], ax=axes[1])

    if on_sem:
        fig, axes = plt.subplots(1,2, figsize=(12,6))
        vmax = max(b2b_sem_mat_sub.max(), b2p_sem_mat.max())
        vmin = min(b2b_sem_mat_sub.min(), b2p_sem_mat.min())
        axes[0].imshow(b2b_sem_mat_sub, cmap='RdYlGn', vmin=vmin, vmax=vmax)
        axes[1].imshow(b2p_sem_mat, cmap='RdYlGn', vmin=vmin, vmax=vmax)
        # Show colorbar 
        fig.colorbar(axes[1].images[0], ax=axes[1])

        cond = np.linalg.cond(b2b_sem_mat_sub)
        print("Condition number:", cond)

        if cond > 1e8:
            print("⚠️ Matrix is severely ill-conditioned")
        elif cond > 1e4:
            print("⚠️ Matrix is moderately ill-conditioned")
        else:
            print("✅ Matrix is relatively stable")


### 

## 7. Save output as raster and data frame

In [ ]:
grid_target[f'{varname}_kriged'] = preds
grid_target[f'kriging_variance'] = preds_var

points_target[f'{varname}_kriged'] = preds2
points_target[f'kriging_variance'] = preds_var2

# Export kriged map to raster (Layer1: kriged values, layer 2: krige variance).
kriged_da = grid_target.set_index(['y', 'x'])[[f'{varname}_kriged', 'kriging_variance']].to_xarray()
kriged_da = kriged_da.rio.write_crs(grid_target.crs)
# kriged_da.attrs.update(grid_target_da.attrs)
buffer_dis = 100
kriged_da_clipped = kriged_da.rio.clip(bound.geometry.buffer(buffer_dis), drop=False, invert=False, all_touched=True)
kriged_da_clipped.rio.to_raster(dir_krig / f'{farm_name}_{date_str}_{varname}_grid{suffix}.tif')


In [ ]:
block_support[f'{varname}_kriged'] = [kriged_da[f'{varname}_kriged'].sel(x=geom.x, y=geom.y, method='nearest').data.item() for geom in block_support.geometry]
block_support_train[f'{varname}_kriged'] = [kriged_da[f'{varname}_kriged'].sel(x=geom.x, y=geom.y, method='nearest').data.item() for geom in block_support_train.geometry]
block_support_test[f'{varname}_kriged'] = [kriged_da[f'{varname}_kriged'].sel(x=geom.x, y=geom.y, method='nearest').data.item() for geom in block_support_test.geometry]
block_support[f'kriging_variance'] = [kriged_da['kriging_variance'].sel(x=geom.x, y=geom.y, method='nearest').data.item() for geom in block_support.geometry]
block_support_train[f'kriging_variance'] = [kriged_da['kriging_variance'].sel(x=geom.x, y=geom.y, method='nearest').data.item() for geom in block_support_train.geometry]
block_support_test[f'kriging_variance'] = [kriged_da['kriging_variance'].sel(x=geom.x, y=geom.y, method='nearest').data.item() for geom in block_support_test.geometry]


#### Export block support dataframe

In [ ]:
block_support_train['x'] = block_support_train.geometry.x
block_support_train['y'] = block_support_train.geometry.y
block_support_train_export = block_support_train.to_crs(epsg=4326)
block_support_train_export['Longitude'] = block_support_train_export.geometry.x
block_support_train_export['Latitude'] = block_support_train_export.geometry.y
block_support_train_export[[varname, 'Longitude', 'Latitude', 'x', 'y', f'{varname}_kriged', 'kriging_variance']].to_csv(
    os.path.join(dir_krig, f'{farm_name}_{date_str}_{varname}_block_support_train{suffix}.csv'), 
    index=False)


In [ ]:
block_support_test['x'] = block_support_test.geometry.x
block_support_test['y'] = block_support_test.geometry.y
block_support_test_export = block_support_test.to_crs(epsg=4326)
block_support_test_export['Longitude'] = block_support_test_export.geometry.x
block_support_test_export['Latitude'] = block_support_test_export.geometry.y
block_support_test_export[[varname, 'Longitude', 'Latitude', 'x', 'y', f'{varname}_kriged', 'kriging_variance']].to_csv(
    dir_krig / f'{farm_name}_{date_str}_{varname}_block_support_test{suffix}.csv', index=False)


In [ ]:
# Free up memory
del block_support_train_export, block_support_test_export

### Control extreme values

In [ ]:
arr_kriged = kriged_da_clipped[f'{varname}_kriged'].data.ravel()
arr_kriged = arr_kriged[~np.isnan(arr_kriged)]
arr_krig_var = kriged_da_clipped['kriging_variance'].data.ravel()
arr_krig_var = arr_krig_var[~np.isnan(arr_krig_var)]

var_p5, var_p95 = np.quantile(arr_krig_var, [0.05, 0.95])
var_p5, var_p95

In [ ]:
block_max, block_min = np.nanmax(block_support[varname]), np.nanmin(block_support[varname])
block_max, block_min
# buffer_counts = 5
kriged_da_masked = kriged_da_clipped.where(kriged_da_clipped['kriging_variance'] < var_p95)
# kriged_da_masked2 = kriged_da_masked.where((block_min - buffer_counts <= kriged_da_masked[f'{varname}_kriged']) & (kriged_da_masked[f'{varname}_kriged'] <= block_max + buffer_counts))

In [ ]:
# Export masked kriged maps to raster
kriged_da_clipped.rio.to_raster(os.path.join(dir_krig, f'{farm_name}_{date_str}_{varname}_grid_clipped{suffix}.tif'))
kriged_da_masked.rio.to_raster(os.path.join(dir_krig, f'{farm_name}_{date_str}_{varname}_grid_masked{suffix}.tif'))
# kriged_da_masked2.rio.to_raster(os.path.join(dir_krig, f'{farm_name}_{date_str}_{varname}_grid_masked2{suffix}.tif'))


### Calculate confidence interval

In [ ]:
from scipy.stats import norm

def calc_confidence_interval(pred, pred_var, confidence=0.95, tail='two'):
    alpha = 1 - confidence
    if tail == 'two':
        z_score = norm.ppf(1 - alpha / 2)
    elif tail == 'one':
        z_score = norm.ppf(alpha)
    else:
        raise ValueError("Invalid tail type. Choose from 'two', or 'one'.")
    lower_bound = pred - z_score * np.sqrt(pred_var)
    upper_bound = pred + z_score * np.sqrt(pred_var)
    return lower_bound, upper_bound

In [ ]:
points_target[['lower_bound', 'upper_bound']] = points_target.apply(lambda x: calc_confidence_interval(x[f'{varname}_kriged'], x['kriging_variance'], confidence=0.95, tail='two'), axis=1).apply(pd.Series)
points_target['block_max'] = block_max
points_target['block_min'] = block_min
points_target['var_p5'] = var_p5
points_target['var_p95'] = var_p95
# buffer_counts = 5
# points_target['flag_outlier'] = points_target.apply(lambda x: (x['kriging_variance'] > x['var_p95']) or (x[f'{varname}_kriged'] < x['block_min']-buffer_counts) or (x[f'{varname}_kriged'] > x['block_max']+buffer_counts), axis=1)


In [ ]:
# Export kriged points to shapefile and csv
# points_target.to_file(os.path.join(pth_out2_krig_sub, f'{farm_name}_{date_str}_{varname}_points{suffix}.shp'))
points_target.drop(['geometry', 'x', 'y'], axis=1
                   ).to_csv(dir_krig /f'{farm_name}_{date_str}_{varname}_points{suffix}.csv', index=False)

## 8. Visulise the results

### 8.1 Map 1: Train-test

In [ ]:
add_contour = False

try:
    dem_proj = rxr.open_rasterio(os.path.join(dirs.lv3_ok, f'{farm_name}_static_DEM_proj.tif'), mask_and_scale=True).squeeze()
except:
    dem_proj = None
    add_contour = False

# Function: add_contour_to_plot. Add contour lines from a shapefile to an existing plot.
def add_contour_to_plot(ax, da, interval=50, colors='Brown', linewidths=1, alpha=1):
    """
    Add contour lines from a shapefile to an existing plot.
    """
    # Read DEM for contour lines
    x = da.x.values
    y = da.y.values
    Z = da.values
    # contour interval
    vmax = np.ceil(np.nanmax(Z) / 100) * 100
    vmin = np.floor(np.nanmin(Z) / 100) * 100
    levels = np.arange(vmin, vmax+0.1, interval)
    # Add contour lines
    cs = ax.contour(x, y, Z, levels=levels, 
            label='Contour lines', colors=colors, linewidths=linewidths, alpha=alpha)
    # Add contour labels
    plt.clabel(cs, inline=True)

if add_contour:
    plt.figure(figsize=(6, 4))
    ax = plt.gca()
    add_contour_to_plot(ax, dem_proj, interval=50)
    # plt.legend()
    plt.show()


In [ ]:
## Map 1: Train-test
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 9), sharex='all', sharey='all')
vmin = block_support[varname].min()
vmax = block_support[varname].max()
# If ploting a raster
kriged_da_masked[f'{varname}_kriged'].plot(ax=axes[1], add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': varname+'_kriged', 'shrink': 0.4}, 
             vmin=vmin, vmax=vmax
            )

# Subplot 0
block_support_test.plot(ax=axes[0], column=f'{varname}', marker='s', edgecolor='black',
                    cmap='Spectral_r', legend=True, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

block_support_train.plot(ax=axes[0], column=f'{varname}', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

# Subplot 1
block_support_test.plot(ax=axes[1], column=f'{varname}', marker='s', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

block_support_train.plot(ax=axes[1], column=f'{varname}', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

axes[0].set_title(f'Sampled {varname}')
axes[1].set_title(f'Kriged {varname} Map')


for ax in axes:
    ax.scatter([0,0], [0,1], color='none', marker='o', edgecolor='black', label='Prediction set')
    ax.scatter([0,0], [0,1], color='none', marker='s', edgecolor='black', label='Evaluation set')
    # ax.legend(loc=1, fontsize=8, bbox_to_anchor=(0.65, 1), frameon=False) # Upper center
    ax.legend(loc='lower left', fontsize=8, bbox_to_anchor=(0.02, 0.02), frameon=True)

    bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
    ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
    ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
    ax.tick_params(labelsize=8, direction='in', length=4)  # Set tick parameters
    ax.set_xlabel('Easting (m)', fontsize=10)  # X label
    ax.set_ylabel('Northing (m)', fontsize=10)  # Y label
    # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box' )  # Set aspect ratio to equal
    
# Add histogram to the upper left corner
if True:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    color = 'deeppink'
    inset_ax1 = inset_axes(axes[0], width="30%", height="20%", borderpad=0, loc=inset_loc)
    inset_ax2 = inset_axes(axes[1], width="30%", height="20%",borderpad=0, loc=inset_loc)

    arr = block_support[varname].values.ravel()
    arr = arr[~np.isnan(arr)]
    inset_ax1.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False, label='All')

    arr1 = block_support_train[varname].values.ravel()
    arr1 = arr1[~np.isnan(arr1)]
    inset_ax1.hist(arr1, bins=20, color='green', alpha=1, edgecolor='white', lw=0.2, density=False, label='Train')

    arr2 = kriged_da_masked[f'{varname}_kriged'].data.ravel()
    arr2 = arr2[~np.isnan(arr2)]
    inset_ax2.hist(arr2, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False, label='Kriged')
    xmin = min(arr.min(), arr2.min())
    xmax = max(arr.max(), arr2.max())
    inset_ax1.set_xlim(xmin, xmax)
    inset_ax2.set_xlim(xmin, xmax)

    for inset_ax in [inset_ax1, inset_ax2]:
        # Set tick parameters
        inset_ax.tick_params(labelsize=6, direction='in', length=2, colors=color)  # Set tick parameters
        # Adjust the position of the y-axis label to right
        inset_ax.yaxis.set_ticks_position('right')  # Y ticks on right
        inset_ax.yaxis.set_label_position('right')  # Y label on to right
        # Set spine colors
        inset_ax.spines['bottom'].set_color(color)
        inset_ax.spines['right'].set_color(color)
        # Set labels and title
        inset_ax.set_xlabel(varname, color=color, fontsize=6, labelpad=1)  # X label
        inset_ax.set_ylabel("Frequency", color=color, fontsize=6, labelpad=1)  # Y label
        inset_ax.set_title('Histogram', fontsize=6, pad=2, color=color)
        # Gridline
        inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
        inset_ax.legend(fontsize=6, loc=2)

plt.suptitle(f'{farm_name} {date_str}\nKriged({mode}) {varname}\n(EPSG:{proj_crs})')
plt.tight_layout()
plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_map1.png'))
if save_pdf:
        plt.savefig(dir_krig_f / f'{farm_name}_{date_str}_{varname}{suffix}_map1.pdf')

plt.show()


### 8.2 Scatter

In [ ]:
from utils.calc_metrics import calc_lccc

In [ ]:
# Drop nan values
plt.figure(figsize=(4, 4))
block_support_train_clean = block_support_train.dropna(subset=[varname, f'{varname}_kriged'], how='any')
block_support_test_clean = block_support_test.dropna(subset=[varname, f'{varname}_kriged'], how='any')
vmin = min(block_support_train_clean[varname].min(), block_support_test_clean[varname].min())
vmax = max(block_support_train_clean[varname].max(), block_support_test_clean[varname].max())

plt.errorbar(block_support_train_clean[varname], block_support_train_clean[f'{varname}_kriged'], 
                yerr=np.sqrt(block_support_train_clean['kriging_variance']), 
                fmt='o', c='green', alpha=0.5, label='Prediction set')
plt.errorbar(block_support_test_clean[varname], block_support_test_clean[f'{varname}_kriged'], 
                yerr=np.sqrt(block_support_test_clean['kriging_variance']),
                fmt='s', c='deeppink', alpha=0.5, label='Evaluation Set')

r_train = np.corrcoef(block_support_train_clean[varname], block_support_train_clean[f'{varname}_kriged'])[0,1]
r_test = np.corrcoef(block_support_test_clean[varname], block_support_test_clean[f'{varname}_kriged'])[0,1]
lccc_train = calc_lccc(block_support_train_clean[varname], block_support_train_clean[f'{varname}_kriged'])
lccc_test = calc_lccc(block_support_test_clean[varname], block_support_test_clean[f'{varname}_kriged'])
rmse_train = np.sqrt(np.mean((block_support_train_clean[varname] - block_support_train_clean[f'{varname}_kriged'])**2))
rmse_test = np.sqrt(np.mean((block_support_test_clean[varname] - block_support_test_clean[f'{varname}_kriged'])**2))
N_train = len(block_support_train_clean)
N_test = len(block_support_test_clean)

ax = plt.gca()
ax.set_aspect('equal', adjustable='box')

fig_text = '\n'.join([f'Kriged vs. Sampled {varname}',
                    f'Train R={r_train:.2f}, Test R={r_test:.2f}',
                    f'Train LCCC={lccc_train:.2f}, Test LCCC={lccc_test:.2f}',
                    f'Train RMSE={rmse_train:.2f}, Test RMSE={rmse_test:.2f}',
                    f'Train N={N_train}, Test N={N_test}'])
ax.text(0.63, 0.05, fig_text,
        transform=ax.transAxes, color='black', verticalalignment='bottom', horizontalalignment='center', fontsize=8,
        bbox=dict(facecolor='none', alpha=0.1, edgecolor='black', boxstyle='square,pad=0.5'))

ax.set_title(f'{farm_name} {date_str}\nKriging of {varname} ({suffix.strip("_test")})')

# plt.title(fig_text)
plt.xlabel('Sampled Value')
plt.ylabel('Kriged Value')
plt.plot([vmin, vmax], [vmin, vmax], ls='--', c='grey')
plt.legend()
plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_scatter.png'))
if save_pdf:
        plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_scatter.pdf'))

plt.show()

# fig, axes = plt.subplots(2, 1, figsize=(2, 4), sharex='all')
# axes[0].hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False, label='All')
# axes[0].hist(arr1, bins=20, color='green', alpha=1, edgecolor='white', lw=0.2, density=False, label='Train')
# axes[1].hist(arr2, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False, label='Kriged')
# axes[0].legend()
# axes[1].legend()
# plt.show()


### 8.3 Variance

In [ ]:
## Map 2: Prediction and variance: Masked by kriging variance at 5-95% range.
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 9), sharex='all', sharey='all')
vmin = block_support[varname].min()
vmax = block_support[varname].max()

kriged_da_masked[f'{varname}_kriged'].plot(ax=axes[0], add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': varname+'_kriged', 'shrink': 0.4}, 
             vmin=vmin, vmax=vmax
            )

# block_support_test.plot(ax=axes[0], column=f'{varname}', marker='s', edgecolor='black',
#                     cmap='Spectral_r', legend=False, # alpha=0.3,
#                     legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
#                     vmin=vmin, vmax=vmax
#                     )

points_target.plot(ax=axes[0], column=f'{varname}_kriged', edgecolor='black', marker='D', markersize=50, zorder=10,
                    cmap='Spectral_r',  legend=False, 
                    legend_kwds={'label': f'{varname}_kriged', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax,
                    )

block_support_train.plot(ax=axes[0], column=f'{varname}', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

if add_contour:   
    add_contour_to_plot(axes[0], dem_proj, interval=20)

axes[0].set_title(f'Kriged {varname}')


vmax = np.percentile(grid_target['kriging_variance'].dropna(), 95)
vmin = np.percentile(grid_target['kriging_variance'].dropna(), 0)

kriged_da_masked[f'kriging_variance'].plot(ax=axes[1], add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': 'kriging_variance', 'shrink': 0.4}, 
             vmin=vmin, vmax=vmax
            )

points_target.plot(ax=axes[1], column=f'kriging_variance', edgecolor='black', marker='D', markersize=50, zorder=10,
                    cmap='Spectral_r',  legend=False, 
                    legend_kwds={'label': f'Kriging variance', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax,
                    )

block_support_train.plot(ax=axes[1], column=f'kriging_variance', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

axes[1].set_title(f'Kriging Variance')

for ax in axes:
    ax.scatter([0,0], [0,1], color='none', marker='o', edgecolor='black', label='Prediction set')
    ax.scatter([0,0], [0,1], color='none', marker='D', edgecolor='black', label='Predicted data')
    # ax.legend(loc=1, fontsize=8, bbox_to_anchor=(0.65, 1), frameon=False) # Upper center
    ax.legend(loc='lower left', fontsize=8, bbox_to_anchor=(0.02, 0.02), frameon=True)

    bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
    ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
    ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
    ax.tick_params(labelsize=8, direction='in', length=4)  # Set tick parameters
    ax.set_xlabel('Easting (m)', fontsize=10)  # X label
    ax.set_ylabel('Northing (m)', fontsize=10)  # Y label
    # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box' )  # Set aspect ratio to equal
    
# Add histogram to the upper left corner
if True:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    color = 'deeppink'
    inset_ax1 = inset_axes(axes[0], width="30%", height="20%", borderpad=0, loc=inset_loc)
    inset_ax2 = inset_axes(axes[1], width="30%", height="20%",borderpad=0, loc=inset_loc)

    arr = kriged_da_masked[f'{varname}_kriged'].data.ravel()
    arr = arr[~np.isnan(arr)]
    
    inset_ax1.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False)

    arr2 = kriged_da_masked['kriging_variance'].data.ravel()
    arr2 = arr2[~np.isnan(arr2)]
    inset_ax2.hist(arr2, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False)

    for inset_ax in [inset_ax1, inset_ax2]:
        # Set tick parameters
        inset_ax.tick_params(labelsize=6, direction='in', length=2, colors=color)  # Set tick parameters
        # Adjust the position of the y-axis label to right
        inset_ax.yaxis.set_ticks_position('right')  # Y ticks on right
        inset_ax.yaxis.set_label_position('right')  # Y label on to right
        # Set spine colors
        inset_ax.spines['bottom'].set_color(color)
        inset_ax.spines['right'].set_color(color)
        # Set labels and title
        inset_ax.set_xlabel(varname, color=color, fontsize=6, labelpad=1)  # X label
        inset_ax.set_ylabel("Frequency", color=color, fontsize=6, labelpad=1)  # Y label
        inset_ax.set_title('Histogram', fontsize=6, pad=2, color=color)
        # Gridline
        inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
        # inset_ax.legend(fontsize=6, loc=2)

plt.suptitle(f'{farm_name} {date_str}\nKriged({mode}) {varname}\n(EPSG:{proj_crs})')
plt.tight_layout()
plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_map2.png'))
if save_pdf:
        plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_map2.pdf'))
plt.show()


In [ ]:
## Map 3: Prediction and variance: Masked by kriging variance at 5-95% range. and block max and min with 1% buffer at both ends
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 9), sharex='all', sharey='all')
vmin = block_support[varname].min()
vmax = block_support[varname].max()

kriged_da_masked[f'{varname}_kriged'].plot(ax=axes[0], add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': varname+'_kriged', 'shrink': 0.4}, 
             vmin=vmin, vmax=vmax
            )

# block_support_test.plot(ax=axes[0], column=f'{varname}', marker='s', edgecolor='black',
#                     cmap='Spectral_r', legend=False, # alpha=0.3,
#                     legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
#                     vmin=vmin, vmax=vmax
#                     )

points_target.plot(ax=axes[0], column=f'{varname}_kriged', edgecolor='black', marker='D', markersize=50, zorder=10,
                    cmap='Spectral_r',  legend=False, 
                    legend_kwds={'label': f'{varname}_kriged', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax,
                    )

block_support_train.plot(ax=axes[0], column=f'{varname}', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

axes[0].set_title(f'Kriged {varname}')


vmax = np.nanmax(kriged_da_masked['kriging_variance'].data.ravel())
vmin = np.nanmin(kriged_da_masked['kriging_variance'].data.ravel())

kriged_da_masked[f'kriging_variance'].plot(ax=axes[1], add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': 'kriging_variance', 'shrink': 0.4}, 
             vmin=vmin, vmax=vmax
            )

points_target.plot(ax=axes[1], column=f'kriging_variance', edgecolor='black', marker='D', markersize=50, zorder=10,
                    cmap='Spectral_r',  legend=False, 
                    legend_kwds={'label': f'Kriging variance', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax,
                    )

block_support_train.plot(ax=axes[1], column=f'kriging_variance', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': varname+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vmax
                    )

axes[1].set_title(f'Kriging Variance')

for ax in axes:
    ax.scatter([0,0], [0,1], color='none', marker='o', edgecolor='black', label='Prediction set')
    ax.scatter([0,0], [0,1], color='none', marker='D', edgecolor='black', label='Predicted data')
    # ax.legend(loc=1, fontsize=8, bbox_to_anchor=(0.65, 1), frameon=False) # Upper center
    ax.legend(loc='lower left', fontsize=8, bbox_to_anchor=(0.02, 0.02), frameon=True)

    bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
    ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
    ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
    ax.tick_params(labelsize=8, direction='in', length=4)  # Set tick parameters
    ax.set_xlabel('Easting (m)', fontsize=10)  # X label
    ax.set_ylabel('Northing (m)', fontsize=10)  # Y label
    # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box' )  # Set aspect ratio to equal
    
# Add histogram to the upper left corner
if True:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    color = 'deeppink'
    inset_ax1 = inset_axes(axes[0], width="30%", height="20%", borderpad=0, loc=inset_loc)
    inset_ax2 = inset_axes(axes[1], width="30%", height="20%",borderpad=0, loc=inset_loc)

    arr = kriged_da_masked[f'{varname}_kriged'].data.ravel()
    arr = arr[~np.isnan(arr)]
    
    inset_ax1.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False)

    arr2 = kriged_da_masked['kriging_variance'].data.ravel()
    arr2 = arr2[~np.isnan(arr2)]
    inset_ax2.hist(arr2, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=False)

    for inset_ax in [inset_ax1, inset_ax2]:
        # Set tick parameters
        inset_ax.tick_params(labelsize=6, direction='in', length=2, colors=color)  # Set tick parameters
        # Adjust the position of the y-axis label to right
        inset_ax.yaxis.set_ticks_position('right')  # Y ticks on right
        inset_ax.yaxis.set_label_position('right')  # Y label on to right
        # Set spine colors
        inset_ax.spines['bottom'].set_color(color)
        inset_ax.spines['right'].set_color(color)
        # Set labels and title
        inset_ax.set_xlabel(varname, color=color, fontsize=6, labelpad=1)  # X label
        inset_ax.set_ylabel("Frequency", color=color, fontsize=6, labelpad=1)  # Y label
        inset_ax.set_title('Histogram', fontsize=6, pad=2, color=color)
        # Gridline
        inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
        # inset_ax.legend(fontsize=6, loc=2)

plt.suptitle(f'{farm_name} {date_str}\nKriged({mode}) {varname}\n(EPSG:{proj_crs})')
plt.tight_layout()
plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_map2_masked.png'))
if save_pdf:
        plt.savefig(os.path.join(dir_krig_f, f'{farm_name}_{date_str}_{varname}{suffix}_map2_masked.pdf'))
plt.show()
